# House Price Prediction

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Data Loading

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True) 
df = housing.frame

df.head()

# EDA

### Dataset Information

In [ ]:
print("Shape:", df.shape)

df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

### Missing Values

In [ ]:
df.isnull().sum()
(df.isnull().mean() * 100).round(2)

### Duplicate Rows

### Separate Numerical & Categorical

In [ ]:
print("Duplicate Rows:", df.duplicated().sum())

In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

print("Numerical Columns:", num_cols)
print("Categorical Columns:", cat_cols)

### Target Variable Analysis

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["MedHouseVal"], kde=True) 
plt.title("Target Distribution") 
plt.show()

### Numerical Feature Distribution

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col], bins=30, kde=True)
    plt.title(f"{col} Distribution")
    plt.show()

### Histogram

### Outlier Detection

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[col])
    plt.title(f"{col} Boxplot")
    plt.show()

### Correlation Matrix

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

### Pair Plot

In [ ]:
sns.pairplot(df)
plt.show()

### Relationship with Target

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.scatterplot(x=df[col], y=df["MedHouseVal"])
    plt.title(f"{col} vs Target")
    plt.show()

# Data Preprocessing

### Feature & Target Split

In [ ]:
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

print("\nFeatures and Target separated")
print("Number of features:", X.shape[1])

### Feature Correlation with Target

In [ ]:
corr = (df.corr()["MedHouseVal"].sort_values(ascending=False))
corr

### Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\nData split into training and testing sets ✅")
print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

### Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Model Training

### Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
print("Training Completed")

y_pred_lr = lr.predict(X_test)
print("Predicted Price:", y_pred_lr)

print("\n=== Linear Regression ===")
print("Features    :", lr.n_features_in_)
print("Coefficients:", lr.coef_)
print("Intercept   :", lr.intercept_)

### Random Forest Regression

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)
rf.fit(X_train, y_train)
print("Training Completed")

y_pred_rf = rf.predict(X_test)
print("Predicted Price:", y_pred_rf)

print("\n=== Random Forest Regression ===")
print("Estimators :", rf.n_estimators)
print("Features   :", rf.n_features_in_)

### Decision Tree Regression

In [ ]:
dt = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)
dt.fit(X_train, y_train)
print("Training Completed")

y_pred_dt = dt.predict(X_test)
print("Predicted Price:", y_pred_dt)

print("\n=== Decision Tree Regression ===")
print("Depth     :", dt.get_depth())
print("Leaves    :", dt.get_n_leaves())
print("Features  :", dt.n_features_in_)

### Lasso Regression

In [ ]:
lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)
print("Training Completed")

y_pred_lasso = lasso.predict(X_test)
print("Predicted Price:", y_pred_lasso)

print("\n=== Lasso Regression ===")
print("Alpha       :", lasso.alpha)
print("Coefficients:", lasso.coef_)

### Ridge Regression

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
print("Training Completed")

y_pred_ridge = ridge.predict(X_test)
print("Predicted Price:", y_pred_ridge)

print("\n=== Ridge Regression ===")
print("Alpha       :", ridge.alpha)
print("Coefficients:", ridge.coef_)

# Model Evaluation

In [ ]:
def evaluate_model(name, y_test, y_pred):
    print(f"\n=== {name} ===")
    print(f"MAE  : {mean_absolute_error(y_test, y_pred):.4f}")
    print(f"MSE  : {mean_squared_error(y_test, y_pred):.4f}")
    print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
    print(f"R2   : {r2_score(y_test, y_pred):.4f}")

In [ ]:
evaluate_model("Linear Regression", y_test, y_pred_lr)
evaluate_model("Decision Tree", y_test, y_pred_dt)
evaluate_model("Random Forest", y_test, y_pred_rf)
evaluate_model("Lasso Regression", y_test, y_pred_lasso)
evaluate_model("Ridge Regression", y_test, y_pred_ridge)

### Model Comparison

In [ ]:
models = ['Linear',
           'RandomForest',
           'DecisionTree',
           'Lasso',
           'Ridge']

scores = [
    r2_score(y_test, y_pred_lr),
    r2_score(y_test, y_pred_rf),
    r2_score(y_test, y_pred_dt),
    r2_score(y_test, y_pred_lasso),
    r2_score(y_test, y_pred_ridge)
]

plt.bar(models, scores)
plt.title("Model Comparison (R2 Score)")
plt.ylabel("Score")
plt.show()

print("\nBest Model:", models[np.argmax(scores)])

# Save Model

In [ ]:
import joblib
joblib.dump(rf, "../saved_models/house_model.pkl")

# Predictions

In [ ]:
sample = X_test[0].reshape(1, -1)
prediction = rf.predict(sample)

print("Predicted Price:", prediction)